## 1. Simple Experiment


### Setup

- Assume a constant individual-level treatment effect. 
- Construct a complete potential-outcomes table for \(2n\) units, where both treatment and control outcomes are known for every individual.
- Enables unlimited resampling via random reassignment of units into treatment and control groups.

### Procedure

For a fixed treatment effect:

1. Randomly assign units to treatment and control groups.
2. Apply outcome capping using a predefined threshold.
3. Conduct a two-sample t-test on the capped outcomes.
4. Record whether the result is statistically significant.
5. Repeat Steps 1–4 many times to estimate the proportion of non-significant results.


### Output Matrix

- Rows: treatment effect sizes  
- Columns: capping thresholds  
- Values: estimated Type II error (or Type I error when effect size = 0)




In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

def run_capping_experiment(df, dataset_name, iterations=1000):
    # Assume group A is the treatment group
    base_revenue = df[df['Group'] == 'A']['REVENUE'].values
    n = len(base_revenue)
    thresholds = [0.90, 0.95, 0.99, 0.999, 1.0] 
    effects = [-10.0, -5.0, -2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 5.0, 10.0] 
    results = []

    # Start simulation
    print(f"Simulating {dataset_name}...")
    for effect in effects:
        row_results = {'Treatment_Effect': effect}
        
        # Set the revenue for the control and the treatment groups
        y0 = base_revenue
        y1 = base_revenue + effect
        
        for q in thresholds:
            significant_count = 0
            
            # Capping value 
            cap_val = np.quantile(y0, q) if q < 1.0 else np.inf
            
            for _ in range(iterations):
                # Resampling
                is_treatment = np.random.rand(n) > 0.5
                
                # Assign the observed revenue to the control and the treatment groups
                observed_rev = np.where(is_treatment, y1, y0)
                
                # Apply Capping on the observed revenue
                capped_rev = np.clip(observed_rev, None, cap_val)
                
                # Split the control and the treatment groups
                group_t = capped_rev[is_treatment]
                group_c = capped_rev[~is_treatment]
                
                # T-test
                if np.var(group_t) == 0 and np.var(group_c) == 0:
                    p_val = 1.0
                else:
                    _, p_val = stats.ttest_ind(group_t, group_c, equal_var=False)
                
                if p_val < 0.05:
                    significant_count += 1
            
            # Calculate Error Rates
            power = significant_count / iterations
            if effect == 0:
                # Type I Error 
                row_results[f'P{int(q*100)}'] = power 
            else:
                # Type II Error 
                row_results[f'P{int(q*100)}'] = 1 - power
        
        results.append(row_results)

    return pd.DataFrame(results)


files = ['Dataset1_outliers_ready.csv', 'Dataset2_outliers_ready.csv', 'Dataset3_outliers_ready.csv']
final_reports = {}

for file in files:
    try:
        data = pd.read_csv(file)
        report = run_capping_experiment(data, file)
        final_reports[file] = report
        
        print(f"\n--- {file} Error Matrix ---")
        print("(Effect 0.0 = Type I Error | Others = Type II Error)")
        print(report.to_string(index=False))
        print("-" * 50)
    except Exception as e:
        print(f"Error processing {file}: {e}")

Starting simulation for Dataset1_outliers_ready.csv...


/Users/denghaonan/opt/anaconda3/envs/merged-env/lib/python3.9/site-packages/scipy/stats/_axis_nan_policy.py:523: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)



--- Dataset1_outliers_ready.csv Error Matrix ---
(Effect 0.0 = Type I Error | Others = Type II Error)
 Treatment_Effect   P90   P95   P99  P100
            -10.0 0.000 0.000 0.000 0.000
             -5.0 0.000 0.000 0.000 0.000
             -2.0 0.000 0.000 0.000 0.033
             -1.0 0.000 0.000 0.296 0.528
             -0.5 0.000 0.000 0.756 0.849
              0.0 0.038 0.042 0.056 0.044
              0.5 0.000 0.000 0.757 0.862
              1.0 0.000 0.000 0.269 0.538
              2.0 0.000 0.000 0.001 0.036
              5.0 0.000 0.000 0.000 0.000
             10.0 0.000 0.000 0.000 0.000
--------------------------------------------------
Starting simulation for Dataset2_outliers_ready.csv...

--- Dataset2_outliers_ready.csv Error Matrix ---
(Effect 0.0 = Type I Error | Others = Type II Error)
 Treatment_Effect   P90   P95   P99  P100
            -10.0 0.000 0.000 0.110 0.096
             -5.0 0.000 0.000 0.619 0.630
             -2.0 0.000 0.209 0.899 0.899
             -1.

/Users/denghaonan/opt/anaconda3/envs/merged-env/lib/python3.9/site-packages/scipy/stats/_axis_nan_policy.py:523: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)



--- Dataset3_outliers_ready.csv Error Matrix ---
(Effect 0.0 = Type I Error | Others = Type II Error)
 Treatment_Effect  P90  P95  P99  P100
            -10.0  0.0  0.0 0.00  0.00
             -5.0  0.0  0.0 0.00  0.00
             -2.0  0.0  0.0 0.00  0.00
             -1.0  0.0  0.0 0.00  0.00
             -0.5  0.0  0.0 0.00  0.00
              0.0  0.0  0.0 0.05  0.05
              0.5  1.0  1.0 0.00  0.00
              1.0  1.0  1.0 0.00  0.00
              2.0  1.0  1.0 0.00  0.00
              5.0  1.0  1.0 0.00  0.00
             10.0  1.0  1.0 0.00  0.00
--------------------------------------------------


In [ ]:
for file_index in range(len(files)):
    final_reports[files[file_index]].to_csv(f'experiment1_Dataset{file_index+1}.csv', index=False)